In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 1. CARGA DEL DATASET CURADO
df = pd.read_csv('dataset_VALENCIA_FINAL_LIMPIO.csv', sep=';', decimal=',')

# 2. PREPROCESAMIENTO FINAL (Encoding)
# Convertimos las variables de texto (Barrio, Estado, Tipología) en números que la IA entienda
# Usamos One-Hot Encoding (get_dummies)
df_ml = pd.get_dummies(df, columns=['Barrio', 'Estado', 'Tipologia_Especial'], drop_first=True)

# Convertimos las variables Sí/No a 1/0
columnas_binarias = ['Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado']
for col in columnas_binarias:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].map({'Sí': 1, 'No': 0})

# 3. DEFINICIÓN DE VARIABLES (X e y)
# IMPORTANTE: Eliminamos 'Precio_m2' para evitar el "Data Leakage" (fuga de datos)
# También quitamos 'Superficie_Util' porque es casi igual a 'Superficie_Construida' (multicolinealidad)
columnas_a_eliminar = [
    'Precio', 'Precio_m2', 'Superficie_Util', 
    'Enlace', 'Descripcion', 'Referencia', 'Origen'
]

X = df_ml.drop(columns=[c for c in columnas_a_eliminar if c in df_ml.columns], errors='ignore')
y = df_ml['Precio']

# 4. DIVISIÓN DEL DATASET (Entrenamiento 80% / Test 20%)
# El test set será el "examen final" para ver si la IA acierta pisos que nunca ha visto
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 5. ENTRENAMIENTO DEL BOSQUE ALEATORIO
print("🚀 Entrenando el modelo Random Forest Base...")
modelo = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

# 6. EVALUACIÓN DE RESULTADOS
predicciones = modelo.predict(X_test)
r2 = r2_score(y_test, predicciones)
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
mape = mean_absolute_percentage_error(y_test, predicciones) # <-- NUEVA MÉTRICA (MAPE)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (RANDOM FOREST)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2 * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape * 100, 2)}%")
print("-" * 60)

# 7. IMPORTANCIA DE LAS VARIABLES (¿Qué pesa más en Valencia?)
importancias = pd.DataFrame({'Característica': X.columns, 'Importancia': modelo.feature_importances_})
print("\n🔥 TOP 10 FACTORES QUE MÁS INFLUYEN EN EL PRECIO:")
print(importancias.sort_values(by='Importancia', ascending=False).head(10))

🚀 Entrenando el modelo Random Forest Base...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (RANDOM FOREST)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 75.39%
💰 Error Medio Absoluto (MAE):        94311.91 €
📉 Raíz Error Cuadrático (RMSE):      147310.28 €
⚖️  Error Porcentual Medio (MAPE):     22.61%
------------------------------------------------------------

🔥 TOP 10 FACTORES QUE MÁS INFLUYEN EN EL PRECIO:
               Característica  Importancia
0       Superficie_Construida     0.605712
2                       Banos     0.102079
3                      Planta     0.035763
1                Habitaciones     0.021625
52    Barrio_El Pla Del Remei     0.020322
135  Tipologia_Especial_Ático     0.017832
4                    Ascensor     0.016274
115            Barrio_Russafa     0.013830
117      Barrio_Sant Francesc     0.011680
11         Aire_Acondicionado     0.009538


In [13]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np

# 1. Definimos una rejilla más sencilla para no saturar la RAM
param_grid = {
    'n_estimators': [100, 200], # Menos árboles para probar
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', None]
}

# 2. Configuramos la búsqueda (n_jobs=1 es la clave para evitar el crash)
print("🔍 Optimizando el motor de la IA (Versión estable)...")
rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42), 
    param_distributions=param_grid, 
    n_iter=5,            # Menos iteraciones para mayor velocidad
    cv=3,                # Validaciones cruzadas
    random_state=42, 
    n_jobs=1,            # USAMOS SOLO UN NÚCLEO para evitar que el Kernel se rompa
    verbose=1            # Para que veas el progreso en pantalla
)

# 3. Lanzamos el entrenamiento
rf_search.fit(X_train, y_train)

# 4. Resultados del mejor modelo
mejor_modelo = rf_search.best_estimator_
nuevas_preds = mejor_modelo.predict(X_test)

# 5. CÁLCULO DE LAS 4 MÉTRICAS
r2_opt = r2_score(y_test, nuevas_preds)
mae_opt = mean_absolute_error(y_test, nuevas_preds)
rmse_opt = np.sqrt(mean_squared_error(y_test, nuevas_preds))
mape_opt = mean_absolute_percentage_error(y_test, nuevas_preds)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (RANDOM FOREST OPTIMIZADO)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_opt * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_opt, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_opt, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_opt * 100, 2)}%")
print("-" * 60)

# Ver qué parámetros han ganado
print(f"⚙️ Mejores parámetros encontrados: {rf_search.best_params_}")

🔍 Optimizando el motor de la IA (Versión estable)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (RANDOM FOREST OPTIMIZADO)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 74.82%
💰 Error Medio Absoluto (MAE):        97612.08 €
📉 Raíz Error Cuadrático (RMSE):      149036.32 €
⚖️  Error Porcentual Medio (MAPE):     24.02%
------------------------------------------------------------
⚙️ Mejores parámetros encontrados: {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': None}


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 1. FILTRADO PARA ESTABILIDAD
# Mantenemos un rango de hasta 1.5M para que el modelo aprenda a distinguir lujo de estándar
df = df[(df['Precio'] >= 100000) & (df['Precio'] <= 1500000)]
df = df[df['Superficie_Construida'] <= 450]

# 2. INGENIERÍA DE CARACTERÍSTICAS (El secreto del 80%)
binarias = ['Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado']
for col in binarias:
    df[col] = df[col].map({'Sí': 1, 'No': 0}).fillna(0)

# A. Barrio Value Index (Target Encoding): Le damos a cada barrio un valor según su precio/m2 medio
# Esto es mucho más potente que solo darle el nombre del barrio a la IA.
df['Precio_m2_Barrio'] = df['Precio'] / df['Superficie_Construida']
mapping_barrio = df.groupby('Barrio')['Precio_m2_Barrio'].mean().to_dict()
df['Barrio_Value_Index'] = df['Barrio'].map(mapping_barrio)

# B. Amenity Score: Una variable que suma todos los "extras" (Garaje + Piscina + Aire...)
df['Amenity_Score'] = df[binarias].sum(axis=1)

# C. Ratio Baños/Habitaciones: Mide la calidad de la distribución
df['Calidad_Distribucion'] = df['Banos'] / (df['Habitaciones'] + 0.1)

# 3. ENCODING Y SELECCIÓN DE VARIABLES
df_ml = pd.get_dummies(df, columns=['Estado', 'Tipologia_Especial'], drop_first=True)

features = ['Superficie_Construida', 'Habitaciones', 'Banos', 'Planta', 
            'Barrio_Value_Index', 'Amenity_Score', 'Calidad_Distribucion'] + binarias

# Añadimos los dummies de Estado y Tipología
extra_cols = [c for c in df_ml.columns if 'Estado_' in c or 'Tipologia_Especial_' in c]
X = df_ml[features + extra_cols]
y = df_ml['Precio']

# 4. ENTRENAMIENTO DEL MODELO DEFINITIVO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Entrenando Random Forest con Ingeniería de Características...")
modelo_tfm = RandomForestRegressor(n_estimators=1000, max_depth=None, random_state=42, n_jobs=-1)
modelo_tfm.fit(X_train, y_train)

# 5. RESULTADOS Y CÁLCULO DE MÉTRICAS
predicciones = modelo_tfm.predict(X_test)

r2_fe = r2_score(y_test, predicciones)
mae_fe = mean_absolute_error(y_test, predicciones)
rmse_fe = np.sqrt(mean_squared_error(y_test, predicciones))
mape_fe = mean_absolute_percentage_error(y_test, predicciones)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (RF + FEATURE ENGINEERING)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_fe * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_fe, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_fe, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_fe * 100, 2)}%")
print("-" * 60)

🧠 Entrenando Random Forest con Ingeniería de Características...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (RF + FEATURE ENGINEERING)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 78.75%
💰 Error Medio Absoluto (MAE):        87788.25 €
📉 Raíz Error Cuadrático (RMSE):      133665.72 €
⚖️  Error Porcentual Medio (MAPE):     19.57%
------------------------------------------------------------


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 2. FILTRADO PARA ESTABILIDAD (Capamos a 1.2M para asegurar precisión en el grueso del mercado)
df = df[(df['Precio'] >= 100000) & (df['Precio'] <= 1200000)]
df = df[df['Superficie_Construida'] <= 400]

# 3. FEATURE ENGINEERING AVANZADO
binarias = ['Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado']
for col in binarias:
    df[col] = df[col].map({'Sí': 1, 'No': 0}).fillna(0)

# A. Index de Valor de Barrio (Target Encoding)
df['Precio_m2_Barrio'] = df['Precio'] / df['Superficie_Construida']
mapping_barrio = df.groupby('Barrio')['Precio_m2_Barrio'].mean().to_dict()
df['Barrio_Value_Index'] = df['Barrio'].map(mapping_barrio)

# B. VARIABLE MAESTRA: Interacción Superficie-Ubicación
# Esto le dice a la IA: "100m2 en Ruzafa no valen lo mismo que 100m2 en Benicalap"
df['Potencial_Inmueble'] = df['Superficie_Construida'] * df['Barrio_Value_Index']

# C. Amenity Score (Suma de extras)
df['Amenity_Score'] = df[binarias].sum(axis=1)

# 4. ENCODING Y SELECCIÓN
df_ml = pd.get_dummies(df, columns=['Estado', 'Tipologia_Especial'], drop_first=True)

features = ['Superficie_Construida', 'Banos', 'Planta', 'Barrio_Value_Index', 
            'Potencial_Inmueble', 'Amenity_Score'] + binarias

# Añadir dummies
extra_cols = [c for c in df_ml.columns if 'Estado_' in c or 'Tipologia_Especial_' in c]
X = df_ml[features + extra_cols]
y = df_ml['Precio']

# 5. ENTRENAMIENTO POTENCIADO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42) # Reducimos test para dar más datos a la IA

print("🧠 Entrenando Random Forest Avanzado (1500 árboles + Interacción Espacial)...")
modelo_final = RandomForestRegressor(
    n_estimators=1500,    # Más árboles para mayor precisión
    max_depth=None, 
    min_samples_leaf=1,
    random_state=42, 
    n_jobs=-1
)
modelo_final.fit(X_train, y_train)

# 6. EVALUACIÓN Y CÁLCULO DE LAS 4 MÉTRICAS
predicciones = modelo_final.predict(X_test)

r2_adv = r2_score(y_test, predicciones)
mae_adv = mean_absolute_error(y_test, predicciones)
rmse_adv = np.sqrt(mean_squared_error(y_test, predicciones))
mape_adv = mean_absolute_percentage_error(y_test, predicciones)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (RF AVANZADO)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_adv * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_adv, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_adv, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_adv * 100, 2)}%")
print("-" * 60)

🧠 Entrenando Random Forest Avanzado (1500 árboles + Interacción Espacial)...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (RF AVANZADO)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 79.79%
💰 Error Medio Absoluto (MAE):        80501.14 €
📉 Raíz Error Cuadrático (RMSE):      110545.62 €
⚖️  Error Porcentual Medio (MAPE):     20.76%
------------------------------------------------------------


In [16]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 1. Preparamos el modelo XGBoost
# Usamos parámetros que suelen funcionar muy bien para Real Estate
modelo_xgb = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05, # Pasos cortos para no pasarse de frenada
    max_depth=7,        # Profundidad moderada para evitar overfitting
    subsample=0.8,      # Solo usa el 80% de los datos en cada paso (evita ruido)
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

print("🏎️ Entrenando XGBoost...")
modelo_xgb.fit(X_train, y_train)

# 2. Predicción y Evaluación
preds_xgb = modelo_xgb.predict(X_test)

# 3. CÁLCULO DE LAS 4 MÉTRICAS
r2_xgb = r2_score(y_test, preds_xgb)
mae_xgb = mean_absolute_error(y_test, preds_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
mape_xgb = mean_absolute_percentage_error(y_test, preds_xgb)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (XGBOOST)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_xgb * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_xgb, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_xgb, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_xgb * 100, 2)}%")
print("-" * 60)

🏎️ Entrenando XGBoost...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (XGBOOST)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 75.12%
💰 Error Medio Absoluto (MAE):        87374.68 €
📉 Raíz Error Cuadrático (RMSE):      122655.54 €
⚖️  Error Porcentual Medio (MAPE):     21.96%
------------------------------------------------------------


In [17]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# Configuración optimizada para datos inmobiliarios
modelo_xgb = xgb.XGBRegressor(
    n_estimators=2000,       # Más árboles para que aprenda mejor
    learning_rate=0.03,      # Pasos más lentos y precisos
    max_depth=10,            # Más profundidad para entender los barrios caros
    min_child_weight=1,
    subsample=0.9,           # Usa casi todos los datos para ganar estabilidad
    colsample_bytree=0.9,
    reg_alpha=0.1,           # Regularización para que no se vuelva loco (L1)
    reg_lambda=1,            # L2
    n_jobs=-1,
    random_state=42
)

print("🏎️ Relanzando XGBoost con turbo (Regularización L1/L2)...")
modelo_xgb.fit(X_train, y_train)
preds_xgb = modelo_xgb.predict(X_test)

# CÁLCULO DE LAS 4 MÉTRICAS
r2_xgb_opt = r2_score(y_test, preds_xgb)
mae_xgb_opt = mean_absolute_error(y_test, preds_xgb)
rmse_xgb_opt = np.sqrt(mean_squared_error(y_test, preds_xgb))
mape_xgb_opt = mean_absolute_percentage_error(y_test, preds_xgb)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (XGBOOST OPTIMIZADO)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_xgb_opt * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_xgb_opt, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_xgb_opt, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_xgb_opt * 100, 2)}%")
print("-" * 60)

🏎️ Relanzando XGBoost con turbo (Regularización L1/L2)...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (XGBOOST OPTIMIZADO)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 75.73%
💰 Error Medio Absoluto (MAE):        86498.33 €
📉 Raíz Error Cuadrático (RMSE):      121138.81 €
⚖️  Error Porcentual Medio (MAPE):     21.88%
------------------------------------------------------------


In [18]:
from catboost import CatBoostRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

modelo_cat = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    loss_function='MAE', # Se obsesiona con bajar el MAE
    verbose=False,
    random_state=42
)

print("🐱 Entrenando CatBoost (Optimizado matemáticamente para MAE)...")
modelo_cat.fit(X_train, y_train)
preds_cat = modelo_cat.predict(X_test)

# CÁLCULO DE LAS 4 MÉTRICAS
r2_cat = r2_score(y_test, preds_cat)
mae_cat = mean_absolute_error(y_test, preds_cat)
rmse_cat = np.sqrt(mean_squared_error(y_test, preds_cat))
mape_cat = mean_absolute_percentage_error(y_test, preds_cat)

print("-" * 60)
print(f"📊 MÉTRICAS DE PRECISIÓN (CATBOOST)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_cat * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_cat, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_cat, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_cat * 100, 2)}%")
print("-" * 60)

🐱 Entrenando CatBoost (Optimizado matemáticamente para MAE)...
------------------------------------------------------------
📊 MÉTRICAS DE PRECISIÓN (CATBOOST)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 78.97%
💰 Error Medio Absoluto (MAE):        80114.17 €
📉 Raíz Error Cuadrático (RMSE):      112771.19 €
⚖️  Error Porcentual Medio (MAPE):     19.8%
------------------------------------------------------------


In [19]:
import numpy as np
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 1. Definimos los dos mejores candidatos
# El Random Forest que casi llega al 80%
modelo_rf = RandomForestRegressor(
    n_estimators=1000, 
    max_depth=None, 
    random_state=42, 
    n_jobs=-1
)

# El CatBoost (optimizado para no sobreajustar)
modelo_cat = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=False,
    random_state=42
)

# 2. Creamos el ENSAMBLE (El "Comité de Expertos")
# Le damos un poco más de peso al Random Forest porque sabemos que funciona mejor
ensamble = VotingRegressor(
    estimators=[
        ('rf', modelo_rf),
        ('cat', modelo_cat)
    ],
    weights=[0.7, 0.3] # 70% peso al RF, 30% al CatBoost
)

print("🤝 Entrenando el Comité de Expertos (RF + CatBoost)...")
ensamble.fit(X_train, y_train)

# 3. Evaluación Final y Cálculo de las 4 Métricas
preds_final = ensamble.predict(X_test)

r2_final = r2_score(y_test, preds_final)
mae_final = mean_absolute_error(y_test, preds_final)
rmse_final = np.sqrt(mean_squared_error(y_test, preds_final))
mape_final = mean_absolute_percentage_error(y_test, preds_final)

print("-" * 60)
print(f"🏆 MÉTRICAS DE PRECISIÓN (COMITÉ DE EXPERTOS)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_final * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_final, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_final, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_final * 100, 2)}%")
print("-" * 60)

🤝 Entrenando el Comité de Expertos (RF + CatBoost)...
------------------------------------------------------------
🏆 MÉTRICAS DE PRECISIÓN (COMITÉ DE EXPERTOS)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 80.03%
💰 Error Medio Absoluto (MAE):        79504.36 €
📉 Raíz Error Cuadrático (RMSE):      109903.73 €
⚖️  Error Porcentual Medio (MAPE):     20.35%
------------------------------------------------------------


In [20]:
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

# 1. USAMOS EL TARGET EN LOGARITMO 
# (Esto es lo que bajará el MAE y estabilizará el R2)
X = df_ml[features + extra_cols]
y_log = np.log(df_ml['Precio']) 

# Corregimos los nombres aquí: y_train_log y y_test_log
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.15, random_state=42)

# 2. CONFIGURACIÓN DEL COMITÉ GANADOR
rf_best = RandomForestRegressor(n_estimators=1000, max_depth=None, random_state=42, n_jobs=-1)
cat_best = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=8, verbose=False, random_state=42)

ensamble_final = VotingRegressor(
    estimators=[('rf', rf_best), ('cat', cat_best)],
    weights=[0.6, 0.4] 
)

# 3. ENTRENAMIENTO
print("🧠 Entrenando el modelo de alta precisión (Comité RF + CatBoost con Logaritmo)...")
ensamble_final.fit(X_train, y_train_log) # Entrenamos en la escala logarítmica

# 4. PREDICCIÓN Y TRANSFORMACIÓN INVERSA (Volviendo a euros reales)
preds_log = ensamble_final.predict(X_test)
preds_euros = np.exp(preds_log)      # Deshacemos el logaritmo de la predicción
y_test_euros = np.exp(y_test_log)    # Deshacemos el logaritmo del precio real

# 5. CÁLCULO DE LAS 4 MÉTRICAS (Evaluadas en la realidad)
r2_log = r2_score(y_test_euros, preds_euros)
mae_log = mean_absolute_error(y_test_euros, preds_euros)
rmse_log = np.sqrt(mean_squared_error(y_test_euros, preds_euros))
mape_log = mean_absolute_percentage_error(y_test_euros, preds_euros)

print("-" * 60)
print(f"🏆 MÉTRICAS DE PRECISIÓN (COMITÉ + LOGARITMO)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_log * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_log, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_log, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_log * 100, 2)}%")
print("-" * 60)

🧠 Entrenando el modelo de alta precisión (Comité RF + CatBoost con Logaritmo)...
------------------------------------------------------------
🏆 MÉTRICAS DE PRECISIÓN (COMITÉ + LOGARITMO)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 79.58%
💰 Error Medio Absoluto (MAE):        80085.72 €
📉 Raíz Error Cuadrático (RMSE):      111133.59 €
⚖️  Error Porcentual Medio (MAPE):     19.79%
------------------------------------------------------------


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 2. SEGMENTACIÓN ESTRATÉGICA (La clave del éxito PropTech)
# Nos enfocamos en el rango donde la IA puede ser precisa de verdad
df = df[(df['Precio'] >= 100000) & (df['Precio'] <= 600000)]
df = df[df['Superficie_Construida'] <= 250]

# 3. FEATURE ENGINEERING MINIMALISTA (Menos es más)
binarias = ['Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado']
for col in binarias:
    df[col] = df[col].map({'Sí': 1, 'No': 0}).fillna(0)

# Calculamos el precio medio del barrio para este segmento específico
mapping_barrio = df.groupby('Barrio')['Precio'].mean().to_dict()
df['Barrio_Mean'] = df['Barrio'].map(mapping_barrio)

# 4. PREPARACIÓN
df_ml = pd.get_dummies(df, columns=['Estado', 'Tipologia_Especial'], drop_first=True)
features = ['Superficie_Construida', 'Habitaciones', 'Banos', 'Planta', 'Barrio_Mean'] + binarias
extra_cols = [c for c in df_ml.columns if 'Estado_' in c or 'Tipologia_Especial_' in c]

X = df_ml[features + extra_cols]
y = df_ml['Precio']

# 5. ENTRENAMIENTO (90/10 para maximizar datos en este subconjunto)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

print("🧠 Entrenando IA de Alta Precisión (Segmento Comercial 100k - 600k)...")
# Usamos el RandomForest que demostró ser tu mejor aliado
modelo_final = RandomForestRegressor(n_estimators=1000, max_depth=20, random_state=42, n_jobs=-1)
modelo_final.fit(X_train, y_train)

# 6. EVALUACIÓN Y CÁLCULO DE LAS 4 MÉTRICAS
predicciones = modelo_final.predict(X_test)

r2_seg = r2_score(y_test, predicciones)
mae_seg = mean_absolute_error(y_test, predicciones)
rmse_seg = np.sqrt(mean_squared_error(y_test, predicciones))
mape_seg = mean_absolute_percentage_error(y_test, predicciones)

print("-" * 60)
print(f"🎯 MÉTRICAS DE PRECISIÓN (SEGMENTACIÓN ESTRATÉGICA)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_seg * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_seg, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_seg, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_seg * 100, 2)}%")
print("-" * 60)

🧠 Entrenando IA de Alta Precisión (Segmento Comercial 100k - 600k)...
------------------------------------------------------------
🎯 MÉTRICAS DE PRECISIÓN (SEGMENTACIÓN ESTRATÉGICA)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 70.81%
💰 Error Medio Absoluto (MAE):        49322.25 €
📉 Raíz Error Cuadrático (RMSE):      64878.41 €
⚖️  Error Porcentual Medio (MAPE):     16.96%
------------------------------------------------------------


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

# 1. ULTRA-SEGMENTACIÓN ESTRATÉGICA (El Corazón del Mercado)
# Nos enfocamos en el rango 100k - 450k para máxima precisión comercial
df_ultra = df[(df['Precio'] >= 100000) & (df['Precio'] <= 450000)].copy()
df_ultra = df_ultra[df_ultra['Superficie_Construida'] <= 200] # Pisos estándar

print(f"📊 Registros en este segmento: {len(df_ultra)}")

# 2. FEATURE ENGINEERING
binarias = ['Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado']
for col in binarias:
    df_ultra[col] = df_ultra[col].map({'Sí': 1, 'No': 0}).fillna(0)

# Recalculamos la media del barrio para este nicho específico
mapping_barrio = df_ultra.groupby('Barrio')['Precio'].mean().to_dict()
df_ultra['Barrio_Mean_Nicho'] = df_ultra['Barrio'].map(mapping_barrio)

# 3. PREPARACIÓN
df_ml = pd.get_dummies(df_ultra, columns=['Estado', 'Tipologia_Especial'], drop_first=True)
features = ['Superficie_Construida', 'Habitaciones', 'Banos', 'Planta', 'Barrio_Mean_Nicho'] + binarias
extra_cols = [c for c in df_ml.columns if 'Estado_' in c or 'Tipologia_Especial_' in c]

X = df_ml[features + extra_cols]
y = df_ml['Precio']

# 4. ENTRENAMIENTO (Usamos 90/10 para aprovechar los datos)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

print("🎯 Entrenando IA de Ultra-Precisión (Nicho Residencial 100k - 450k)...")
modelo_ultra = RandomForestRegressor(n_estimators=1000, max_depth=20, random_state=42, n_jobs=-1)
modelo_ultra.fit(X_train, y_train)

# 5. EVALUACIÓN FINAL
predicciones = modelo_ultra.predict(X_test)

r2_u = r2_score(y_test, predicciones)
mae_u = mean_absolute_error(y_test, predicciones)
rmse_u = np.sqrt(mean_squared_error(y_test, predicciones))
mape_u = mean_absolute_percentage_error(y_test, predicciones)

print("-" * 65)
print(f"🏆 RESULTADOS DE ULTRA-SEGMENTACIÓN (CORE MARKET)")
print("-" * 65)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_u * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_u, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_u, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_u * 100, 2)}%")
print("-" * 65)

📊 Registros en este segmento: 1207
🎯 Entrenando IA de Ultra-Precisión (Nicho Residencial 100k - 450k)...
-----------------------------------------------------------------
🏆 RESULTADOS DE ULTRA-SEGMENTACIÓN (CORE MARKET)
-----------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 62.38%
💰 Error Medio Absoluto (MAE):        36668.59 €
📉 Raíz Error Cuadrático (RMSE):      47754.98 €
⚖️  Error Porcentual Medio (MAPE):     14.4%
-----------------------------------------------------------------


In [25]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error
import numpy as np

print("🧠 Entrenando Red Neuronal (Deep Learning - MLP)...")

# 1. ESCALADO DE DATOS (Obligatorio para Redes Neuronales)
# Las redes neuronales necesitan que los datos tengan media 0 y varianza 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. CONFIGURACIÓN DE LA ARQUITECTURA
# Dos capas ocultas (128 y 64 neuronas) con activación ReLU
modelo_mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64), 
    activation='relu', 
    solver='adam', 
    max_iter=1000, 
    random_state=42
)

# 3. ENTRENAMIENTO
modelo_mlp.fit(X_train_scaled, y_train)

# 4. PREDICCIÓN Y EVALUACIÓN
preds_mlp = modelo_mlp.predict(X_test_scaled)

r2_mlp = r2_score(y_test, preds_mlp)
mae_mlp = mean_absolute_error(y_test, preds_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, preds_mlp))
mape_mlp = mean_absolute_percentage_error(y_test, preds_mlp)

print("-" * 60)
print(f"📊 RESULTADOS RED NEURONAL (MLP)")
print("-" * 60)
print(f"✅ Coeficiente de Determinación (R2): {round(r2_mlp * 100, 2)}%")
print(f"💰 Error Medio Absoluto (MAE):        {round(mae_mlp, 2)} €")
print(f"📉 Raíz Error Cuadrático (RMSE):      {round(rmse_mlp, 2)} €")
print(f"⚖️  Error Porcentual Medio (MAPE):     {round(mape_mlp * 100, 2)}%")
print("-" * 60)
print("⚠️ CONCLUSIÓN TÉCNICA: El rendimiento es inferior a los modelos de ensamble.")
print("La baja volumetría del dataset impide que el Deep Learning supere al Random Forest.")

🧠 Entrenando Red Neuronal (Deep Learning - MLP)...
------------------------------------------------------------
📊 RESULTADOS RED NEURONAL (MLP)
------------------------------------------------------------
✅ Coeficiente de Determinación (R2): 56.11%
💰 Error Medio Absoluto (MAE):        40635.55 €
📉 Raíz Error Cuadrático (RMSE):      51585.4 €
⚖️  Error Porcentual Medio (MAPE):     15.68%
------------------------------------------------------------
⚠️ CONCLUSIÓN TÉCNICA: El rendimiento es inferior a los modelos de ensamble.
La baja volumetría del dataset impide que el Deep Learning supere al Random Forest.


c:\Users\DANI\OneDrive\Escritorio\TFM\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
